# Analisis de datos tomografia

Frecuencia de estudios. 

Promedio CTDI.vol y DLP - Gráficas de estas dos cantidades en gráfico de cajones.

Seccionar por pesos de pacientes y establecer rangos de peso para graficar promedios de CTDI y DLP. 

También graficar el 3er cuartil. 

In [130]:
#carga los datos y analiza los datos de dosis en tomografia
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import re



# Cargar datos
#leer los 5 archivos en la carpeta de Datos cargar y solo la hoja llamada "Datos" de cada uno
df = pd.read_excel('Datos_tomo/Datos_Tomografia_HUN.xlsx')

#eliminamos las columnas que son unnamed
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [131]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9843 entries, 0 to 9842
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Paciente             9843 non-null   object        
 1   Procedimento Medico  9843 non-null   object        
 2   Modalidad            9843 non-null   object        
 3   Tecnologo            9843 non-null   object        
 4   Paciente ID          9843 non-null   object        
 5   Numero de Acceso     9843 non-null   int64         
 6   Sexo                 9843 non-null   object        
 7   Entidad              9843 non-null   object        
 8   Fecha de la Toma     9843 non-null   datetime64[ns]
 9   Dosis Radiación      4692 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 769.1+ KB


In [132]:
df

,Paciente,Procedimento Medico,Modalidad,Tecnologo,Paciente ID,Numero de Acceso,Sexo,Entidad,Fecha de la Toma,Dosis Radiación
0,"ABAUNZA DE GONZALEZ,MYRIAM",RADIOGRAFÍA DINÁMICA DE COLUMNA VERTEBRAL,CR,Jenny Morales Quiroga,41596930CC,1519559,Female,UNISALUD - UNIVERSIDAD NACIONAL DE COLOMBIA M...,2025-07-01 09:19:17.885,\n** Estudio: Lumbar **\nPeso del paciente: 58...
1,"ABAUNZA DE GONZALEZ,MYRIAM",RADIOGRAFIA PANORAMICA DE COLUMNA (GONIOMETRIA...,CR,Jenny Morales Quiroga,41596930CC,1519560,Female,UNISALUD - UNIVERSIDAD NACIONAL DE COLOMBIA M...,2025-07-01 09:19:17.948,* Estudio: Goniometría Lumbar **\nPeso del pac...
2,"ABELLO DE DIAZ,FLOR MARINA",ARTERIOGRAFIA DE VASOS ABDOMINALES (SELECTIVA),XA,Alba Carolina Suarez Perdomo,20792667CC,1526674,Female,FIDUPREVISORA S.A.,2025-07-31 11:33:40.917,NaN
3,"ABELLO DE DIAZ,FLOR MARINA",DRENAJE DE COLECCIÓN INTRAPERITONEAL VÍA PERCU...,XA,Alexander Jimenez Ruiz,20792667CC,1518939,Female,FIDUPREVISORA S.A.,2025-06-27 14:26:36.981,NaN
4,"ABELLO DE DIAZ,FLOR MARINA",DRENAJE DE COLECCIÓN INTRAPERITONEAL VÍA PERCU...,XA,Jhoan Sebastian Franco Garces,20792667CC,1518984,Female,FIDUPREVISORA S.A.,2025-06-27 14:08:23.744,NaN
...,...,...,...,...,...,...,...,...,...,...
9838,"ZULUAGA FLOREZ,FRANCISCO JOSE JAVIER",RADIOGRAFIA DE PIE AP Y LATERAL,CR,Andrea Yanuba Mendez Rondon,19307950CC,1532895,Male,SANITAS EPS,2025-08-25 19:57:10.227,Peso del paciente: 68 Kg\nDist: Bucky; Pos: AP...
9839,"ZULUAGA TAMAYO,ALBERT",RADIOGRAFIA DE TORAX (PAO APY LATERAL DECUBITO...,CR,Juan Pablo Restrepo Vargas,70909063CC,1533724,Male,EPS SURA,2025-08-28 14:06:47.054,"""** Estudio: Tórax **\nPeso del paciente: 70 K..."
9840,"ZUÑIGA GUEVARA,ARIANA VICTORIA",RADIOGRAFIA DE TORAX (PAO APY LATERAL DECUBITO...,CR,Andrea Yanuba Mendez Rondon,1005678615CC,1520344,Female,EPS SURA,2025-07-03 18:46:44.843,** Estudio: Tórax **\nPeso del paciente: 80 Kg...
9841,"ZUÑIGA GUEVARA,ARIANA VICTORIA",TOMOGRAFIA AXIAL COMPUTADA DE ABDOMEN Y PELVI...,CT,Yuly Patricia Alfonso Fuquen,1005678615CC,1520827,Female,EPS SURA,2025-07-06 16:47:21.763,"Peso 92\nCTDI.vol 12,3\nDLP 711,3\nContraste S..."


### Limpieza

In [133]:
#eliminamos las filas que tienen NAN en las columna de Dosis Radiación
df = df[df['Dosis Radiación'].notna()]
#nos quedamos solo con modalidad de CT
df = df[df['Modalidad'] == 'CT']

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2232 entries, 45 to 9842
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Paciente             2232 non-null   object        
 1   Procedimento Medico  2232 non-null   object        
 2   Modalidad            2232 non-null   object        
 3   Tecnologo            2232 non-null   object        
 4   Paciente ID          2232 non-null   object        
 5   Numero de Acceso     2232 non-null   int64         
 6   Sexo                 2232 non-null   object        
 7   Entidad              2232 non-null   object        
 8   Fecha de la Toma     2232 non-null   datetime64[ns]
 9   Dosis Radiación      2232 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 191.8+ KB


In [134]:
# Contar ocurrencias por procedimiento
counts = df['Procedimento Medico'].value_counts()
# Filtrar procedimientos con más de 170 ocurrencias

proceds_filtrados = counts[counts >= 170].index
df_filtrado = df[df['Procedimento Medico'].isin(proceds_filtrados)]

# Histograma solo para procedimientos con conteo > 170
fig = px.histogram(df_filtrado, x='Procedimento Medico', title='Número de datos por procedimiento (>170)')
fig.update_layout(bargap=0.2)
fig.show()

In [135]:
#quitamos TOMOGRAFÍA COMPUTADA DE VASOS','TOMOGRAFIA AXIAL COMPUTADA DE SENOS PARANASALES O CARA (CORTES AXIALES Y CORONALES)
df_filtrado= df_filtrado[~df_filtrado['Procedimento Medico'].isin(['TOMOGRAFÍA COMPUTADA DE VASOS','TOMOGRAFIA AXIAL COMPUTADA DE SENOS PARANASALES O CARA (CORTES AXIALES Y CORONALES)'])]

#cOLOCAMOS LOS STRING DE LA COLUMNA DE DOSIS RADIACIÓN EN Minuscula
df_filtrado['Dosis Radiación'] = df_filtrado['Dosis Radiación'].str.lower()
df_filtrado['Dosis Radiación'] = df_filtrado['Dosis Radiación'].str.strip()
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1251 entries, 45 to 9841
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Paciente             1251 non-null   object        
 1   Procedimento Medico  1251 non-null   object        
 2   Modalidad            1251 non-null   object        
 3   Tecnologo            1251 non-null   object        
 4   Paciente ID          1251 non-null   object        
 5   Numero de Acceso     1251 non-null   int64         
 6   Sexo                 1251 non-null   object        
 7   Entidad              1251 non-null   object        
 8   Fecha de la Toma     1251 non-null   datetime64[ns]
 9   Dosis Radiación      1251 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 107.5+ KB


### * Separacion de la informacion en la columna Dosis Radiación

In [136]:
df_filtrado['Dosis Radiación'].unique()

array(['peso 91\nctdi.vol 12,9\ndlp 804,1\ncontraste si\nml/300mol 90',
       'peso 70\nctdi.vol 17,5\ndlp 877\ncontraste si\ncantidad 75',
       'peso 65\nctdi.vol 11,5\ndlp 674,8\ncontraste si\nml/300mol 60\nesatura 160',
       ..., 'peso 85\nctdi.vol 4,9\ndlp 201\ncontraste si\nml/300mol 80',
       'peso 75\nctdi.vol 41.70\ndlp 983\ncontraste no\ncantidad 0',
       'peso 92\nctdi.vol 12,3\ndlp 711,3\ncontraste si\nml/300mol 80'],
      dtype=object)

In [137]:
# Contar cuántos registros en 'Dosis Radiación' tienen la estructura exacta tipo:
# 'peso ...\nctdi.vol ...\ndlp ...\ncontraste ...' (en cualquier orden, pero todas presentes)
def tiene_estructura_estandar(s):
    if not isinstance(s, str):
        return False
    s = s.lower()
    return all(clave in s for clave in ['peso', 'ctdi.vol', 'dlp', 'contraste'])

n_estructura = df_filtrado['Dosis Radiación'].apply(tiene_estructura_estandar).sum()
print(f"Número de registros con estructura estándar: {n_estructura}")

Número de registros con estructura estándar: 1087


In [138]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1251 entries, 45 to 9841
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Paciente             1251 non-null   object        
 1   Procedimento Medico  1251 non-null   object        
 2   Modalidad            1251 non-null   object        
 3   Tecnologo            1251 non-null   object        
 4   Paciente ID          1251 non-null   object        
 5   Numero de Acceso     1251 non-null   int64         
 6   Sexo                 1251 non-null   object        
 7   Entidad              1251 non-null   object        
 8   Fecha de la Toma     1251 non-null   datetime64[ns]
 9   Dosis Radiación      1251 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 107.5+ KB


In [139]:

def extraer_hasta_contraste(df, col="Dosis Radiación"):
    allowed_norm = {"peso", "ctdi_vol", "dlp", "contraste"}

    def norm_key(s: str) -> str:
        s = (s or "").strip()
        s = s.replace("CTDI.vol", "CTDI_vol")          # unifica
        s = re.sub(r"[^A-Za-z0-9]+", "_", s)           # separadores -> _
        s = re.sub(r"_+", "_", s).strip("_")
        return s.casefold()

    def parse_num(s: str):
        s = (s or "").strip().replace(",", ".")
        m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
        return float(m.group(0)) if m else None

    def parse_bool(s: str):
        t = (s or "").strip().casefold()
        if t in {"si", "sí", "yes", "true"}:  return True
        if t in {"no", "false"}:               return False
        return None

    registros = []
    for raw in df[col].fillna(""):
        out = {"Peso": None, "CTDI_vol": None, "DLP": None, "Contraste": None}
        stop = False

        for line in str(raw).splitlines():
            if stop: break
            line = line.strip()
            if not line: continue

            # --- 1) k:v
            if ":" in line:
                k, v = line.split(":", 1)

            else:
                # --- 2) "k v"
                parts = line.split(None, 1)
                if len(parts) == 2:
                    k, v = parts
                else:
                    # --- 3) "kNNN" (sin espacio): ej. "peso70", "dlp217.90"
                    m = re.match(r"^([A-Za-z][A-Za-z0-9_.\/]*)\s*([-+]?\d+(?:[.,]\d+)?)$", line)
                    # también admite sin espacio entre clave y número:
                    if not m:
                        m = re.match(r"^([A-Za-z][A-Za-z0-9_.\/]*?)([-+]?\d+(?:[.,]\d+)?)$", line)
                    if m:
                        k, v = m.group(1), m.group(2)
                    else:
                        continue  # fuera de estructura → ignorar

            k_norm = norm_key(k)
            if k_norm not in allowed_norm:
                continue

            if k_norm == "peso":
                val = parse_num(v)
                if val is not None: out["Peso"] = val

            elif k_norm == "ctdi_vol":
                val = parse_num(v)
                if val is not None: out["CTDI_vol"] = val

            elif k_norm == "dlp":
                val = parse_num(v)
                if val is not None: out["DLP"] = val

            elif k_norm == "contraste":
                b = parse_bool(v)
                if b is not None: out["Contraste"] = b
                stop = True  # "hasta Contraste"

        registros.append(out)

    parsed = pd.DataFrame(registros)
    return pd.concat([df.reset_index(drop=True), parsed], axis=1)


df_filtrado = extraer_hasta_contraste(df_filtrado, col="Dosis Radiación")

In [140]:
df_filtrado.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1251 entries, 0 to 1250
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Paciente             1251 non-null   object        
 1   Procedimento Medico  1251 non-null   object        
 2   Modalidad            1251 non-null   object        
 3   Tecnologo            1251 non-null   object        
 4   Paciente ID          1251 non-null   object        
 5   Numero de Acceso     1251 non-null   int64         
 6   Sexo                 1251 non-null   object        
 7   Entidad              1251 non-null   object        
 8   Fecha de la Toma     1251 non-null   datetime64[ns]
 9   Dosis Radiación      1251 non-null   object        
 10  Peso                 1238 non-null   float64       
 11  CTDI_vol             1239 non-null   float64       
 12  DLP                  1240 non-null   float64       
 13  Contraste            1083 non-nul

In [141]:
#lo guardamos en un nuevo archivo excel
df_filtrado.to_excel('Datos_tomo/Datos_Tomografia_HUN_Procesado.xlsx', index=False)

### Graficas con solo 4 procedimientos ya limpiados

In [145]:
#graficamos el promedio de dosis por procedimiento medico
# Unificamos los nombres de los procedimientos de tórax
df_filtrado['Procedimento Medico'] = df_filtrado['Procedimento Medico'].replace(
    {
        'TOMOGRAFÍA COMPUTADA DE TÓRAX DE ALTA RESOLUCIÓN (TCAR)': 'TOMOGRAFIA AXIAL COMPUTADA DE TORAX'
    }
)

# Calculamos el promedio de DLP por procedimiento médico (ya unificados)
promedios = df_filtrado.groupby('Procedimento Medico').agg({'DLP': 'mean'}).reset_index()
fig = go.Figure(data=[
    go.Bar(name='DLP', x=promedios['Procedimento Medico'], y=promedios['DLP'])
])
fig.update_layout(
    barmode='group',
    title='Promedio de Dosis por Procedimiento Médico',
    xaxis_title='Procedimiento Médico',
    yaxis_title='Dosis Promedio'
)

